# Cleaning & Preparation Dataset Berita Politik (Detik) untuk Sentiment Analysis IndoBERT

Notebook ini:
- Membaca CSV mentah
- Menghasilkan dataset final dengan kolom **date, title, content, text, label**
- Menjaga kompatibilitas untuk **IndoBERT** dan kemungkinan **IndoBERT + BiLSTM**
- Menjaga kolom tanggal agar siap untuk **agregasi time series (Tahap 2)**

> Input: `detik_articles.csv`

> Output:
- `detik_clean_for_indobert.csv`
- `detik_labeling_template.csv` (untuk anotasi manual)
- `detik_clean_with_heuristic_label.csv` (opsional: label awal berbasis lexicon/keyword)

In [7]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import os

# Input CSV (gunakan path absolut sandbox jika ada; fallback ke file di folder kerja)
INPUT_PATH = Path("../../raw_fixed/detik_articles.csv")
if not INPUT_PATH.exists():
    INPUT_PATH = Path("detik_articles.csv")

OUTPUT_CLEAN = Path("detik_clean_for_indobert.csv")
OUTPUT_LABEL_TEMPLATE = Path("detik_labeling_template.csv")
OUTPUT_HEURISTIC = Path("detik_clean_with_heuristic_label.csv")

df_raw = pd.read_csv(INPUT_PATH)
print("Shape:", df_raw.shape)
df_raw.tail(3)

Shape: (9999, 4)


,url,title,published_at,content
9996,https://news.detik.com/pemilu/d-6909357/relawa...,"Relawan Ganjar Gelar Lomba Mural, Ajak Anak Mu...","Sabtu, 02 Sep 2023 13:39 WIB",Tim Koordinasi Relawan Pemenangan Pilpres PDIP...
9997,https://news.detik.com/pemilu/d-6909350/prabow...,Prabowo Tegaskan Tak Akan Impor Energi Bila Ja...,"Sabtu, 02 Sep 2023 13:31 WIB",Ketua Umum Partai Gerindra Prabowo Subianto me...
9998,https://news.detik.com/pemilu/d-6909344/prabow...,Prabowo soal Pogram Jokowi: Yang Benar Kita Te...,"Sabtu, 02 Sep 2023 13:25 WIB",Bakal capres sekaligus Ketum Partai Gerindra P...


## 1) Standarisasi kolom
Dataset mentah umumnya memiliki kolom: `title`, `published_at` (tanggal string), `content`.

Notebook ini akan:
- Mengambil hanya `title`, `published_at`, `content`
- Membuat kolom `published_datetime` (datetime)
- Membuat kolom `date` (YYYY-MM-DD) untuk agregasi time series

In [16]:
# --- 1.1 Select required columns ---
required_candidates = {
    "title": ["title", "judul"],
    "published_at": ["published_at", "date", "published", "published_time", "waktu", "tanggal"],
    "content": ["content", "isi", "body", "article", "text"]
}

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

col_title = pick_col(df_raw, required_candidates["title"])
col_date  = pick_col(df_raw, required_candidates["published_at"])
col_cont  = pick_col(df_raw, required_candidates["content"])

if not all([col_title, col_date, col_cont]):
    raise ValueError(f"Kolom tidak lengkap. Ditemukan: title={col_title}, date={col_date}, content={col_cont}. Kolom tersedia: {list(df_raw.columns)}")

df = df_raw[[col_title, col_date, col_cont]].copy()
df.columns = ["title", "published_at", "content"]

df.head(3)

,title,published_at,content
0,Komisi I DPR Dukung TNI Berbenah Usai 17 Praju...,"Kamis, 01 Jan 2026 08:53 WIB",Sebanyak 17 prajurit TNI dari Yonif Teritorial...
1,Saat SBY Terganggu karena Kena Fitnah Isu Ijaz...,"Kamis, 01 Jan 2026 07:49 WIB",Isu tudingan ijazah palsu Presiden ke-7 RI Jok...
2,7 Capaian Polda Metro Sepanjang 2025,"Kamis, 01 Jan 2026 06:02 WIB",Kepolisian Daerah (Polda) Metro Jaya menutup t...


## 2) Parsing tanggal Detik
Format yang umum dari Detik:

`Kamis, 01 Jan 2026 08:53 WIB`

Kita ubah menjadi:
- `published_datetime`: `2026-01-01 08:53:00`
- `date`: `2026-01-01`

In [17]:
MONTH_MAP = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "Mei": 5, "Jun": 6,
    "Jul": 7, "Agu": 8, "Sep": 9, "Okt": 10, "Nov": 11, "Des": 12
}

DATE_RE = re.compile(
    r"(?:(?:Senin|Selasa|Rabu|Kamis|Jumat|Sabtu|Minggu),\s*)?"
    r"(\d{1,2})\s+(Jan|Feb|Mar|Apr|Mei|Jun|Jul|Agu|Sep|Okt|Nov|Des)\s+(\d{4})"
    r"(?:\s+(\d{1,2}):(\d{2}))?"
)

def parse_detik_datetime(s: str):
    if pd.isna(s):
        return pd.NaT
    s = str(s).strip()
    m = DATE_RE.search(s)
    if not m:
        return pd.NaT
    day = int(m.group(1))
    mon = MONTH_MAP[m.group(2)]
    year = int(m.group(3))
    hh = int(m.group(4)) if m.group(4) else 0
    mm = int(m.group(5)) if m.group(5) else 0
    return pd.Timestamp(year=year, month=mon, day=day, hour=hh, minute=mm)

df["published_datetime"] = df["published_at"].apply(parse_detik_datetime)
df["date"] = df["published_datetime"].dt.date.astype("datetime64[ns]")  # normalize to midnight

missing = df["published_datetime"].isna().mean()
print(f"Missing parsed datetime: {missing:.2%}")
df[["published_at", "published_datetime", "date"]].head(5)

Missing parsed datetime: 0.17%


,published_at,published_datetime,date
0,"Kamis, 01 Jan 2026 08:53 WIB",2026-01-01 08:53:00,2026-01-01
1,"Kamis, 01 Jan 2026 07:49 WIB",2026-01-01 07:49:00,2026-01-01
2,"Kamis, 01 Jan 2026 06:02 WIB",2026-01-01 06:02:00,2026-01-01
3,"Rabu, 31 Des 2025 17:48 WIB",2025-12-31 17:48:00,2025-12-31
4,"Rabu, 31 Des 2025 17:27 WIB",2025-12-31 17:27:00,2025-12-31


## 3) Cleaning teks (IndoBERT-friendly)
Prinsip:
- **Tidak** melakukan stemming/stopword removal.
- Hapus noise (URL, HTML tag, spasi berlebih).
- Jaga kata negasi (tidak/bukan) dan struktur kalimat.

Output:
- `content_clean`
- `title_clean`
- `text` = `title_clean + '. ' + content_clean`

In [18]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
HTML_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"\s+")
SYMBOL_RE = re.compile(r"[^a-zA-Z0-9\s\.,!?\'\-]")

def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s)
    s = HTML_RE.sub(" ", s)
    s = URL_RE.sub(" ", s)
    s = SYMBOL_RE.sub(" ", s)
    s = s.lower()
    s = WS_RE.sub(" ", s).strip()
    return s


df["title_clean"] = df["title"].apply(clean_text)
df["content_clean"] = df["content"].apply(clean_text)
df["text"] = (df["title_clean"].str.strip() + ". " + df["content_clean"].str.strip()).str.strip()
df[["title", "title_clean", "content_clean"]].head(3)

,title,title_clean,content_clean
0,Komisi I DPR Dukung TNI Berbenah Usai 17 Praju...,komisi i dpr dukung tni berbenah usai 17 praju...,sebanyak 17 prajurit tni dari yonif teritorial...
1,Saat SBY Terganggu karena Kena Fitnah Isu Ijaz...,saat sby terganggu karena kena fitnah isu ijaz...,isu tudingan ijazah palsu presiden ke-7 ri jok...
2,7 Capaian Polda Metro Sepanjang 2025,7 capaian polda metro sepanjang 2025,kepolisian daerah polda metro jaya menutup tah...


## 4) Filtering kualitas dokumen
Kita buang:
- konten terlalu pendek (default: < 50 kata)
- duplikat (`text` sama)
- baris tanpa tanggal valid

Catatan: threshold kata bisa Anda ubah sesuai inspeksi data.

In [19]:
MIN_WORDS = 50

def word_count(s: str) -> int:
    if not isinstance(s, str) or not s:
        return 0
    return len(s.split())

df["word_count"] = df["content_clean"].apply(word_count)

before = len(df)
df = df.dropna(subset=["published_datetime"])
df = df[df["word_count"] >= MIN_WORDS].copy()
df = df.drop_duplicates(subset=["text"]).copy()
after = len(df)

print(f"Rows before: {before:,} -> after filtering: {after:,} (dropped {(before-after):,})")
df[["date", "title_clean", "word_count"]].head(5)

Rows before: 9,999 -> after filtering: 9,643 (dropped 356)


,date,title_clean,word_count
0,2026-01-01,komisi i dpr dukung tni berbenah usai 17 praju...,397
1,2026-01-01,saat sby terganggu karena kena fitnah isu ijaz...,1051
2,2026-01-01,7 capaian polda metro sepanjang 2025,1159
3,2025-12-31,demokrat somasi akun tiktok terkait tuduhan sb...,363
4,2025-12-31,kilas balik peristiwa penting di detikcom year...,338


## 5) Kolom final untuk modelling + time series
Kolom yang dipertahankan:
- `date`
- `title` (clean)
- `content` (clean)
- `text` (gabungan)
- `label` (kosong untuk anotasi manual)

In [20]:
df_final = df[["date", "title_clean", "content_clean", "text"]].copy()
df_final = df_final.rename(columns={"title_clean": "title", "content_clean": "content"})
df_final["label"] = np.nan  # isi manual: -1 negatif, 0 netral, 1 positif

df_final.head(5)

,date,title,content,text,label
0,2026-01-01,komisi i dpr dukung tni berbenah usai 17 praju...,sebanyak 17 prajurit tni dari yonif teritorial...,komisi i dpr dukung tni berbenah usai 17 praju...,NaN
1,2026-01-01,saat sby terganggu karena kena fitnah isu ijaz...,isu tudingan ijazah palsu presiden ke-7 ri jok...,saat sby terganggu karena kena fitnah isu ijaz...,NaN
2,2026-01-01,7 capaian polda metro sepanjang 2025,kepolisian daerah polda metro jaya menutup tah...,7 capaian polda metro sepanjang 2025. kepolisi...,NaN
3,2025-12-31,demokrat somasi akun tiktok terkait tuduhan sb...,partai demokrat resmi melayangkan somasi ke sa...,demokrat somasi akun tiktok terkait tuduhan sb...,NaN
4,2025-12-31,kilas balik peristiwa penting di detikcom year...,tahun 2025 akan segera tutup buku. sepanjang t...,kilas balik peristiwa penting di detikcom year...,NaN


## SEED DATASET for MANUAL LABELLING

In [22]:
import os

SEED_PATH = "seed_labeling_manual.csv"

# Pastikan ada row_id yang stabil
df_final = df_final.copy()
df_final["row_id"] = range(len(df_final))

# Ambil seed berdasarkan row_id
df_seed = df_final.sample(n=1000, random_state=42)

# Unlabeled = semua yang row_id-nya tidak ada di seed
df_unlabeled = df_final[~df_final["row_id"].isin(df_seed["row_id"])]

# (Opsional) drop row_id kalau Anda tidak mau kolom ini ikut ke model
# Tapi saya sarankan disimpan untuk tracking dan audit.
if os.path.exists(SEED_PATH):
    os.remove(SEED_PATH)

df_seed.to_csv(SEED_PATH, index=False)

# Reset index hanya untuk rapih
df_seed = df_seed.reset_index(drop=True)
df_unlabeled = df_unlabeled.reset_index(drop=True)


In [24]:
display(df_seed.tail())
display(df_unlabeled.tail())

,date,title,content,text,label,row_id
995,2023-09-27,waka mpr alkhairaat harus jadi kekuatan islam ...,wakil ketua mpr fadel muhammad yang juga merup...,waka mpr alkhairaat harus jadi kekuatan islam ...,NaN,9117
996,2025-09-20,bos ppi nilai tak wajar gibran absen pelantika...,waketum projo freddy damanik menanggapi direkt...,bos ppi nilai tak wajar gibran absen pelantika...,NaN,763
997,2025-05-16,membaca maksud jokowi kalkulasi peluang jadi k...,partai solidaritas indonesia psi telah membuka...,membaca maksud jokowi kalkulasi peluang jadi k...,NaN,1783
998,2023-12-04,gaya gibran blusukan di jakarta saat akhir pekan,"cawapres nomor urut 2, gibran rakabuming raka ...",gaya gibran blusukan di jakarta saat akhir pek...,NaN,7613
999,2025-05-21,puan minta negara hadir atasi fenomena phk tak...,ketua dpr ri puan maharani menyoroti maraknya ...,puan minta negara hadir atasi fenomena phk tak...,NaN,1731


,date,title,content,text,label,row_id
8638,2023-09-02,yusril yakin psi dukung prabowo meski belum de...,ketua umum ketum partai bulan bintang pbb yusr...,yusril yakin psi dukung prabowo meski belum de...,NaN,9637
8639,2023-09-02,fadli zon jamin koalisi prabowo tak terganggu ...,waketum partai gerindra fadli zon menghadiri k...,fadli zon jamin koalisi prabowo tak terganggu ...,NaN,9639
8640,2023-09-02,"relawan ganjar gelar lomba mural, ajak anak mu...",tim koordinasi relawan pemenangan pilpres pdip...,"relawan ganjar gelar lomba mural, ajak anak mu...",NaN,9640
8641,2023-09-02,prabowo tegaskan tak akan impor energi bila ja...,ketua umum partai gerindra prabowo subianto me...,prabowo tegaskan tak akan impor energi bila ja...,NaN,9641
8642,2023-09-02,prabowo soal pogram jokowi yang benar kita ter...,bakal capres sekaligus ketum partai gerindra p...,prabowo soal pogram jokowi yang benar kita ter...,NaN,9642


## 6) (Opsional) Heuristic pre-label untuk mempercepat anotasi
Ini **bukan** label final penelitian, tetapi label awal berbasis keyword.

Rekomendasi praktis:
- Gunakan heuristic ini untuk membuat prioritas anotasi.
- Tetap lakukan anotasi manual dan/atau validasi inter-annotator untuk label final.

Skema label:
-1 = negatif, 0 = netral, 1 = positif

In [8]:
POS_KWS = [
    "stabil", "meningkat", "apresiasi", "dukungan", "optimis", "optimistis",
    "penguatan", "perbaikan", "kepercayaan", "damai", "kondusif", "berhasil"
]
NEG_KWS = [
    "krisis", "gejolak", "konflik", "gagal", "korupsi", "demo", "kerusuhan",
    "ketidakpastian", "tertekan", "melemah", "turun", "skandal", "ricuh"
]

def heuristic_label(text: str) -> int:
    t = text or ""
    pos = sum(1 for w in POS_KWS if w in t)
    neg = sum(1 for w in NEG_KWS if w in t)
    if pos == 0 and neg == 0:
        return 0
    if pos > neg:
        return 1
    if neg > pos:
        return -1
    return 0  # tie -> netral

df_heur = df_final.copy()
df_heur["heuristic_label"] = df_heur["text"].apply(heuristic_label)

df_heur["heuristic_label"].value_counts(dropna=False)

heuristic_label
 0    3766
 1    3496
-1    2381
Name: count, dtype: int64

## 7) Simpan output
1) Dataset bersih untuk IndoBERT (tanpa heuristic)
2) Template anotasi manual (hanya kolom kunci)
3) Dataset bersih + heuristic label (opsional)

In [9]:
df_final.to_csv(OUTPUT_CLEAN, index=False, encoding="utf-8")
df_final[["date", "title", "content", "text", "label"]].to_csv(OUTPUT_LABEL_TEMPLATE, index=False, encoding="utf-8")
df_heur.to_csv(OUTPUT_HEURISTIC, index=False, encoding="utf-8")

print("Saved:")
print("-", OUTPUT_CLEAN.resolve())
print("-", OUTPUT_LABEL_TEMPLATE.resolve())
print("-", OUTPUT_HEURISTIC.resolve())

Saved:
- D:\File\Private\OneDrive - Bina Nusantara\Kuliah\s k r i p s i s\code\source code\detik_clean_for_indobert.csv
- D:\File\Private\OneDrive - Bina Nusantara\Kuliah\s k r i p s i s\code\source code\detik_labeling_template.csv
- D:\File\Private\OneDrive - Bina Nusantara\Kuliah\s k r i p s i s\code\source code\detik_clean_with_heuristic_label.csv


In [4]:
temp = pd.read_csv("detik_clean_for_indobert.csv")
temp.tail()

,date,title,content,text,label
9638,2023-09-02,fadli zon ungkap gerindra terbuka untuk demokr...,partai demokrat merasa dikhianati usai duet an...,fadli zon ungkap gerindra terbuka untuk demokr...,NaN
9639,2023-09-02,fadli zon jamin koalisi prabowo tak terganggu ...,waketum partai gerindra fadli zon menghadiri k...,fadli zon jamin koalisi prabowo tak terganggu ...,NaN
9640,2023-09-02,"relawan ganjar gelar lomba mural, ajak anak mu...",tim koordinasi relawan pemenangan pilpres pdip...,"relawan ganjar gelar lomba mural, ajak anak mu...",NaN
9641,2023-09-02,prabowo tegaskan tak akan impor energi bila ja...,ketua umum partai gerindra prabowo subianto me...,prabowo tegaskan tak akan impor energi bila ja...,NaN
9642,2023-09-02,prabowo soal pogram jokowi yang benar kita ter...,bakal capres sekaligus ketum partai gerindra p...,prabowo soal pogram jokowi yang benar kita ter...,NaN


## 8) Cek kesiapan agregasi time series (Tahap 2)
Contoh agregasi sentimen harian (setelah label final terisi):
- mean(label) per tanggal
- jumlah berita per tanggal

Ini akan dipakai untuk korelasi/regresi dengan volatilitas LQ45.

In [ ]:
# Contoh (setelah label terisi):
# df_labeled = pd.read_csv(OUTPUT_LABEL_TEMPLATE)
# df_labeled["date"] = pd.to_datetime(df_labeled["date"])
# daily = df_labeled.dropna(subset=["label"]).groupby("date").agg(
#     sentiment_mean=("label", "mean"),
#     news_count=("label", "size")
# ).reset_index()
# daily.head()

print("Notebook selesai. Isi kolom 'label' pada file template untuk training.")